In [ ]:
!pip install transformers torch emoji nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 42.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import glob
import re
import emoji
import nltk
from transformers import pipeline

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
files = glob.glob("sample_data/datasets/*.csv")

df_list = []

for file in files:
    df = pd.read_csv(file, engine="python", on_bad_lines="skip")
    df_list.append(df)
    print("Loaded:", file)

print("Total datasets:", len(df_list))


Loaded: sample_data/datasets/covid19_tweets.csv
Loaded: sample_data/datasets/MEST-CoV-test.csv
Loaded: sample_data/datasets/METS-CoV-dev.csv
Loaded: sample_data/datasets/METS-CoV-train.csv
Loaded: sample_data/datasets/Covid-19 Twitter Dataset (Apr-Jun 2020).csv
Loaded: sample_data/datasets/TweetDataSA.csv
Loaded: sample_data/datasets/Australia.csv
Total datasets: 7


In [ ]:
def standardize_columns(df):

    column_map = {
        "text": "Tweet",
        "original_text": "Tweet",
        "Tweets": "Tweet",

        "created_at": "Date",
        "date": "Date",
        "DateCreated": "Date",

        "user_location": "Location",
        "place": "Location",
        "Location": "Location"
    }

    df = df.rename(columns=column_map)

    for col in ["Tweet", "Date", "Location"]:
        if col not in df.columns:
            df[col] = None

    return df[["Tweet", "Date", "Location"]]

clean_dfs = [standardize_columns(df) for df in df_list]

merged_df = pd.concat(clean_dfs, ignore_index=True)

merged_df.dropna(subset=["Tweet"], inplace=True)
merged_df.drop_duplicates(inplace=True)

print("Merged dataset shape:", merged_df.shape)
merged_df.head()

Merged dataset shape: (87648, 3)


,Tweet,Date,Location
0,If I smelled the scent of hand sanitizers toda...,2020-07-25 12:27:21,astroworld
1,Hey @Yankees @YankeesPR and @MLB - wouldn't it...,2020-07-25 12:27:17,"New York, NY"
2,@diane3443 @wdunlap @realDonaldTrump Trump nev...,2020-07-25 12:27:14,"Pewee Valley, KY"
3,@brookbanktv The one gift #COVID19 has give me...,2020-07-25 12:27:10,Stuck in the Middle
4,25 July : Media Bulletin on Novel #CoronaVirus...,2020-07-25 12:27:08,Jammu and Kashmir


In [ ]:
def clean_tweet(text):

    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r"\s+", " ", text).strip()

    return text

merged_df["Clean_Tweet"] = merged_df["Tweet"].apply(clean_tweet)

merged_df.head()

,Tweet,Date,Location,Clean_Tweet
0,If I smelled the scent of hand sanitizers toda...,2020-07-25 12:27:21,astroworld,if i smelled the scent of hand sanitizers toda...
1,Hey @Yankees @YankeesPR and @MLB - wouldn't it...,2020-07-25 12:27:17,"New York, NY",hey and - wouldn't it have made more sense to ...
2,@diane3443 @wdunlap @realDonaldTrump Trump nev...,2020-07-25 12:27:14,"Pewee Valley, KY",trump never once claimed covid19 was a hoax. w...
3,@brookbanktv The one gift #COVID19 has give me...,2020-07-25 12:27:10,Stuck in the Middle,the one gift covid19 has give me is an appreci...
4,25 July : Media Bulletin on Novel #CoronaVirus...,2020-07-25 12:27:08,Jammu and Kashmir,25 july : media bulletin on novel coronavirusu...


In [ ]:
aspects = {
    "vaccine": ["vaccine", "vaccination", "booster"],
    "lockdown": ["lockdown", "quarantine"],
    "hospital": ["hospital", "doctor", "health"],
    "government": ["government", "policy"],
    "economy": ["economy", "jobs", "business"]
}

def detect_aspects(text):

    found = []

    for aspect, keywords in aspects.items():
        if any(word in text for word in keywords):
            found.append(aspect)

    if not found:
        found.append("general")

    return found

In [ ]:
sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [ ]:
# Limit dataset size for faster execution
merged_df = merged_df.sample(50000, random_state=42)

print("Sampled dataset size:", merged_df.shape)

Sampled dataset size: (50000, 4)


In [ ]:
batch_size = 32
results = []

texts = merged_df["Clean_Tweet"].tolist()

for i in range(0, len(texts), batch_size):

    batch = texts[i:i+batch_size]

    sentiments = sentiment_model(batch)

    for j, sentiment in enumerate(sentiments):

        row = merged_df.iloc[i + j]
        detected_aspects = detect_aspects(row["Clean_Tweet"])

        for aspect in detected_aspects:

            results.append({
                "Tweet": row["Tweet"],
                "Aspect": aspect,
                "Sentiment": sentiment["label"],
                "Date": row["Date"],
                "Location": row["Location"]
            })

    print(f"Processed {min(i + batch_size, len(texts))} tweets...")


Processed 32 tweets...
Processed 64 tweets...
Processed 96 tweets...
Processed 128 tweets...
Processed 160 tweets...
Processed 192 tweets...
Processed 224 tweets...
Processed 256 tweets...
Processed 288 tweets...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Processed 320 tweets...
Processed 352 tweets...
Processed 384 tweets...
Processed 416 tweets...
Processed 448 tweets...
Processed 480 tweets...
Processed 512 tweets...
Processed 544 tweets...
Processed 576 tweets...
Processed 608 tweets...
Processed 640 tweets...
Processed 672 tweets...
Processed 704 tweets...
Processed 736 tweets...
Processed 768 tweets...
Processed 800 tweets...
Processed 832 tweets...
Processed 864 tweets...
Processed 896 tweets...
Processed 928 tweets...
Processed 960 tweets...
Processed 992 tweets...
Processed 1024 tweets...
Processed 1056 tweets...
Processed 1088 tweets...
Processed 1120 tweets...
Processed 1152 tweets...
Processed 1184 tweets...
Processed 1216 tweets...
Processed 1248 tweets...
Processed 1280 tweets...
Processed 1312 tweets...
Processed 1344 tweets...
Processed 1376 tweets...
Processed 1408 tweets...
Processed 1440 tweets...
Processed 1472 tweets...
Processed 1504 tweets...
Processed 1536 tweets...
Processed 1568 tweets...
Processed 1600 tweets.

In [ ]:
absa_df = pd.DataFrame(results)

print("Final dataset shape:", absa_df.shape)
absa_df.head()

Final dataset shape: (52092, 5)


,Tweet,Aspect,Sentiment,Date,Location
0,@heidigarand @Gab_H_R Source: https://t.co/kec...,general,LABEL_1,Wed Dec 08 13:01:07 +0000 2021,"Western Australia, Australia"
1,"#COVID19 update for Africa, 25 July 2020 @ 9am...",general,LABEL_1,2020-07-25 09:53:21,Utrecht
2,RT @ever_relentless: The government is asking ...,lockdown,LABEL_0,2020-04-26,Chicago IL
3,RT @ever_relentless: The government is asking ...,hospital,LABEL_0,2020-04-26,Chicago IL
4,RT @ever_relentless: The government is asking ...,government,LABEL_0,2020-04-26,Chicago IL


In [ ]:
absa_df.to_csv("absa_output.csv", index=False)

print("ABSA dataset saved!")

ABSA dataset saved!


In [ ]:
summary = absa_df.groupby(
    ["Aspect", "Sentiment"]
).size().reset_index(name="Count")

summary.to_csv("summary_powerbi.csv", index=False)

summary

,Aspect,Sentiment,Count
0,economy,LABEL_0,375
1,economy,LABEL_1,375
2,economy,LABEL_2,91
3,general,LABEL_0,10119
4,general,LABEL_1,17147
5,general,LABEL_2,4009
6,government,LABEL_0,989
7,government,LABEL_1,943
8,government,LABEL_2,87
9,hospital,LABEL_0,1405


In [ ]:
label_map = {
    "LABEL_0": "Negative",
    "LABEL_1": "Neutral",
    "LABEL_2": "Positive"
}

absa_df["Sentiment"] = absa_df["Sentiment"].map(label_map)

absa_df.head()

,Tweet,Aspect,Sentiment,Date,Location
0,@heidigarand @Gab_H_R Source: https://t.co/kec...,general,Neutral,Wed Dec 08 13:01:07 +0000 2021,"Western Australia, Australia"
1,"#COVID19 update for Africa, 25 July 2020 @ 9am...",general,Neutral,2020-07-25 09:53:21,Utrecht
2,RT @ever_relentless: The government is asking ...,lockdown,Negative,2020-04-26,Chicago IL
3,RT @ever_relentless: The government is asking ...,hospital,Negative,2020-04-26,Chicago IL
4,RT @ever_relentless: The government is asking ...,government,Negative,2020-04-26,Chicago IL


In [ ]:
absa_df.to_csv("absa_output.csv", index=False)

In [ ]:
absa_df["Location"] = absa_df["Location"].fillna("Unknown")

In [ ]:
absa_df["Date"] = pd.to_datetime(absa_df["Date"], errors="coerce")

/tmp/ipython-input-2084801507.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  absa_df["Date"] = pd.to_datetime(absa_df["Date"], errors="coerce")
/tmp/ipython-input-2084801507.py:1: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  absa_df["Date"] = pd.to_datetime(absa_df["Date"], errors="coerce")
